# 🔷 Session 1 & 2 — Introduction to OpenVINO

**Intel DevMeet 1.0 Nagpur — Text to Image Generation with Qwen using OpenVINO**

---

This notebook covers the foundational concepts of the **Intel OpenVINO Toolkit** and walks through a basic tutorial on model optimization and inference.

> 📌 This notebook is intended to run on **Google Colab**.

## 📑 Table of Contents

1. [What is Intel OpenVINO?](#what-is-openvino)
2. [AI Inference Acceleration & Optimization](#inference-acceleration)
3. [Supported Frameworks & Deployment Pipeline](#supported-frameworks)
4. [Install Dependencies](#install-dependencies)
5. [Model Optimization Workflow](#model-optimization)
6. [Converting a Model to OpenVINO IR](#converting-model)
7. [Running Inference with OpenVINO Runtime](#running-inference)
8. [Summary](#summary)

## 1. What is Intel OpenVINO? <a id='what-is-openvino'></a>

[Intel OpenVINO™](https://www.intel.com/content/www/us/en/developer/tools/openvino-toolkit/overview.html) (Open Visual Inference & Neural Network Optimization) is an open-source toolkit for optimizing and deploying AI inference on Intel hardware (CPUs, GPUs, VPUs, and FPGAs) as well as heterogeneous execution environments.

### Key Features
- ⚡ **High-performance inference** on Intel CPUs, integrated GPUs, and neural compute sticks.
- 🔄 **Framework agnostic** — supports models from PyTorch, TensorFlow, ONNX, PaddlePaddle, and more.
- 🛠️ **Model Optimizer** — converts and optimizes trained models into OpenVINO's Intermediate Representation (IR).
- 📦 **Compact deployment** — reduced model size and memory footprint for edge devices.
- 🌐 **Cross-platform** — runs on Linux, Windows, and macOS.

## 2. AI Inference Acceleration & Optimization <a id='inference-acceleration'></a>

OpenVINO accelerates inference through several mechanisms:

| Technique | Description |
|---|---|
| **INT8 Quantization** | Reduces model weights from FP32 to INT8, drastically cutting memory and compute. |
| **Layer Fusion** | Combines consecutive operations (e.g., Conv + BN + ReLU) into a single kernel. |
| **Graph Pruning** | Removes unused nodes from the computation graph. |
| **Hardware-aware Scheduling** | Dispatches compute to the optimal device (CPU, iGPU, etc.). |
| **Asynchronous Inference** | Overlaps pre/post-processing with inference for higher throughput. |

## 3. Supported Frameworks & Deployment Pipeline <a id='supported-frameworks'></a>

```
Trained Model (PyTorch / TensorFlow / ONNX / PaddlePaddle)
        │
        ▼
  [ OpenVINO Model Optimizer / OVC ]
        │
        ▼
  OpenVINO IR (.xml + .bin)
        │
        ▼
  [ OpenVINO Runtime (Core, CompiledModel) ]
        │
        ▼
  Inference on Target Hardware
  (CPU / Intel GPU / Neural Compute Stick / FPGA)
```

## 4. Install Dependencies <a id='install-dependencies'></a>

Run the cell below to install the required packages.

In [ ]:
# Install OpenVINO and supporting libraries
%pip install -q openvino openvino-dev[pytorch,onnx] torch torchvision pillow matplotlib

In [ ]:
import openvino as ov
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print(f"OpenVINO version: {ov.__version__}")
print(f"PyTorch version:  {torch.__version__}")

## 5. Model Optimization Workflow <a id='model-optimization'></a>

The OpenVINO optimization workflow involves three key steps:

1. **Export** the trained model from the source framework (e.g., PyTorch → ONNX).
2. **Convert** the model to OpenVINO IR format using `ov.convert_model()` or the `ovc` CLI tool.
3. **Compile** the IR model for the target hardware and run inference.

Let's walk through this using a simple ResNet-18 model as an example.

In [ ]:
import torchvision.models as models

# Load a pretrained ResNet-18 model from PyTorch
pytorch_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
pytorch_model.eval()
print("ResNet-18 loaded successfully.")

## 6. Converting a Model to OpenVINO IR <a id='converting-model'></a>

Use `ov.convert_model()` to convert the PyTorch model directly — no intermediate ONNX export needed.

In [ ]:
# Define a representative input shape: batch=1, channels=3, height=224, width=224
dummy_input = torch.zeros(1, 3, 224, 224)

# Convert the PyTorch model to OpenVINO IR
ov_model = ov.convert_model(pytorch_model, example_input=dummy_input)
print("Model converted to OpenVINO IR.")

# (Optional) Save the IR model to disk
ov.save_model(ov_model, "resnet18.xml")
print("IR model saved as resnet18.xml + resnet18.bin")

## 7. Running Inference with OpenVINO Runtime <a id='running-inference'></a>

We use the OpenVINO `Core` class to compile and run the model on the available device.

In [ ]:
# List available inference devices
core = ov.Core()
print("Available devices:", core.available_devices)

In [ ]:
# Compile the model for CPU inference
compiled_model = core.compile_model(ov_model, device_name="CPU")

# Create a random input tensor (simulating an image batch)
input_data = np.random.rand(1, 3, 224, 224).astype(np.float32)

# Run inference
output = compiled_model([input_data])
logits = output[0]  # shape: (1, 1000) for ImageNet classes

predicted_class = np.argmax(logits[0])
print(f"Predicted ImageNet class index: {predicted_class}")

### ⏱️ Benchmark: Measuring Inference Latency

In [ ]:
import time

N_RUNS = 100
start = time.perf_counter()
for _ in range(N_RUNS):
    compiled_model([input_data])
elapsed = time.perf_counter() - start

print(f"Average inference latency over {N_RUNS} runs: {elapsed / N_RUNS * 1000:.2f} ms")

## 8. Summary <a id='summary'></a>

In this notebook you learned:

- ✅ What Intel OpenVINO is and why it matters for AI deployment.
- ✅ The key acceleration techniques OpenVINO applies (quantization, fusion, pruning).
- ✅ Which frameworks are supported and the general deployment pipeline.
- ✅ How to convert a PyTorch model to OpenVINO IR format.
- ✅ How to compile an OpenVINO IR model and run inference on the CPU.

---

➡️ **Next:** Open `02_Text_to_Image_Generation_with_Qwen_OpenVINO.ipynb` for the hands-on Text-to-Image workshop.